In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [4]:
# =============================================
# DL-YourRollNo-notebook-t22026
# Smart MCQ Solver - Full Pipeline + LoRA
# =============================================

import pandas as pd
import numpy as np
import torch
from tqdm import tqdm
import wandb
from huggingface_hub import login

print("✅ Libraries Imported")

# ============================
# 1. LOGIN & W&B
# ============================

# ============================
# 2. LOAD DATA
# ============================

train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

print(f"Train: {train_df.shape} | Test: {test_df.shape}")

# ============================
# 3. PREPROCESSING
# ============================

def clean_text(text):
    return str(text).strip()

for col in ['prompt', 'A', 'B', 'C', 'D', 'E']:
    train_df[col] = train_df[col].apply(clean_text)
    if col in test_df.columns:
        test_df[col] = test_df[col].apply(clean_text)

print("✅ Data Cleaned")

✅ Libraries Imported
Train: (2000, 8) | Test: (500, 7)
✅ Data Cleaned


In [8]:
import torch
import gc
import os

# Memory Optimization Settings
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.cuda.empty_cache()
gc.collect()

print("Memory optimization enabled")

Memory optimization enabled


In [5]:
# ============================
# 4. BASELINE EMBEDDINGS
# ============================

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

emb_model = SentenceTransformer('all-MiniLM-L6-v2')

def baseline_top3(row):
    q_emb = emb_model.encode(row['prompt'])
    opts = [row['A'], row['B'], row['C'], row['D'], row['E']]
    opt_emb = emb_model.encode(opts)
    sims = cosine_similarity([q_emb], opt_emb)[0]
    top3_idx = np.argsort(sims)[-3:][::-1]
    labels = ['A','B','C','D','E']
    return ' '.join([labels[i] for i in top3_idx])

test_df['baseline_pred'] = test_df.apply(baseline_top3, axis=1)
print("✅ Baseline Done")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Baseline Done


In [6]:
# ============================
# 5. IMPROVED SCORING FUNCTION (Model 2)
# ============================

from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True
)

def improved_score_option(prompt, option, label=""):
    full_prompt = f"""Question: {prompt}
Option {label}: {option}
Is this the best answer? Reason step by step and give confidence score (0-100):"""
    
    inputs = tokenizer(full_prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model(**inputs)
        # Better scoring: use mean of last few tokens
        score = outputs.logits[0, -5:].mean().item()
    return score

def pretrained_top3(row):
    prompt = row['prompt']
    options = {'A': row['A'], 'B': row['B'], 'C': row['C'], 'D': row['D'], 'E': row['E']}
    scores = {k: improved_score_option(prompt, v, k) for k, v in options.items()}
    sorted_opts = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return ' '.join([x[0] for x in sorted_opts[:3]])

# Run on sample first (for speed)
sample_test = test_df.head(30).copy()
sample_test['pretrained_pred'] = sample_test.apply(pretrained_top3, axis=1)
print("✅ Improved Pretrained Scoring Done")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

✅ Improved Pretrained Scoring Done


In [9]:
# ============================
# 6. LoRA FINE-TUNING (Memory Optimized)
# ============================

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import BitsAndBytesConfig, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from datasets import Dataset
import torch

# Clear memory
torch.cuda.empty_cache()
gc.collect()

# Use 4-bit quantization (much lighter)
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map="auto",
    trust_remote_code=True
)

# Prepare for LoRA
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,                    # Reduced from 16
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],  # Fewer modules
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Format dataset
def format_for_finetune(example):
    prompt = f"""Question: {example['prompt']}
A) {example['A']}
B) {example['B']}
C) {example['C']}
D) {example['D']}
E) {example['E']}
Correct Answer: {example['answer']}"""
    return {"text": prompt}

train_dataset = Dataset.from_pandas(train_df.sample(frac=0.6))  # Use 60% data to save memory
train_dataset = train_dataset.map(format_for_finetune)

# Training args - very memory friendly
training_args = TrainingArguments(
    output_dir="./qwen_lora_mcqa",
    per_device_train_batch_size=1,      # Critical for memory
    gradient_accumulation_steps=16,     # Effective batch = 16
    num_train_epochs=1,
    learning_rate=3e-4,
    fp16=True,
    logging_steps=10,
    report_to="wandb",
    save_strategy="no",
    optim="adamw_8bit"                  # 8-bit optimizer
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
)

trainer.train()
print("✅ LoRA Fine-tuning Completed (Memory Optimized)!")

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

trainable params: 2,523,136 || all params: 7,618,139,648 || trainable%: 0.0331


Map:   0%|          | 0/1200 [00:00<?, ? examples/s]

ValueError: No columns in the dataset match the model's forward method signature: (input_ids, attention_mask, position_ids, past_key_values, inputs_embeds, labels, use_cache, cache_position, logits_to_keep, kwargs, label_ids, label, labels). The following columns have been ignored: [prompt, B, text, answer, C, D, id, __index_level_0__, E, A]. Please check the dataset and model. You may need to set `remove_unused_columns=False` in `TrainingArguments`.

In [ ]:
# ============================
# 7. ENSEMBLE + FINAL SUBMISSION
# ============================

def ensemble(row):
    votes = {}
    for pred_col in ['baseline_pred', 'pretrained_pred']:
        if pred_col in row:
            for rank, letter in enumerate(str(row[pred_col]).split()[:3]):
                votes[letter] = votes.get(letter, 0) + (3 - rank)
    sorted_pred = sorted(votes.items(), key=lambda x: x[1], reverse=True)
    return ' '.join([x[0] for x in sorted_pred[:3]])

test_df['Prediction'] = test_df.apply(ensemble, axis=1)

submission = test_df[['id', 'Prediction']].copy()
submission.rename(columns={'id': 'ID'}, inplace=True)
submission.to_csv('submission.csv', index=False)

print("✅ Final Submission Ready!")
display(submission.head())